In [2]:
import gymnasium as gym 
import random 
import torch 
import numpy  as np
import matplotlib.pyplot as plt 
from collections import  deque , namedtuple
import math

In [3]:
# env define import from gymnasium  
env = gym.make("CartPole-v1" , render_mode='human')
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"

In [4]:
Experience = namedtuple('Experience', ['state' , 'action' , 'reward' , 'next_state' , 'done'])

class ReplayMemory() :
    def __init__(self , capacity):
        self.capacity = capacity
        self.memory = deque([] , maxlen=self.capacity)

    def push(self ,*args) :
        self.memory.append(Experience(*args))

    def sample(self , batchSize=128) :
        return   random.sample(self.memory , batchSize)

    def __len__(self):
        return len(self.memory)

In [5]:
# neural network def 
class  DQN(torch.nn.Module):
    def __init__(self , obs, actions ) :
        super(DQN , self).__init__()
        self.layerStack = torch.nn.Sequential(
            torch.nn.Linear(obs , 128 ) ,
            torch.nn.ReLU() ,
            torch.nn.Linear(128 , 128 ) ,
            torch.nn.ReLU() ,
            torch.nn.Linear(128,actions)
        )
    def forward(self , x) :
         return  self.layerStack(x)

     


In [6]:
BATCH_SIZE = 128
GAMMA = 0.99
EPS_START = 0.9
EPS_END = 0.01
EPS_DECAY = 2500
TAU = 0.005
LR = 3e-4


# Get number of actions from gym action space
n_actions = env.action_space.n
# Get the number of state observations
state, info = env.reset()
n_observations = len(state)

policy_net = DQN(n_observations, n_actions).to(device)
target_net = DQN(n_observations, n_actions).to(device)
target_net.load_state_dict(policy_net.state_dict())

optimizer = torch.optim.AdamW(policy_net.parameters(), lr=LR, amsgrad=True)
memory = ReplayMemory(10000)

steps_done = 0 

def select_action(state):
    global steps_done
    sample = random.random()
    eps_threshold = EPS_END + (EPS_START - EPS_END) * \
        math.exp(-1. * steps_done / EPS_DECAY)
    steps_done += 1
    if sample > eps_threshold:
        with torch.no_grad():
            return policy_net(state).max(1).indices.view(1, 1)
    else:
        return torch.tensor([[env.action_space.sample()]], device=device, dtype=torch.long)

episode_durations = []

def plot_durations(show_result=False):
    plt.figure(1)
    durations_t = torch.tensor(episode_durations, dtype=torch.float)
    if show_result:
        plt.title('Result')
    else:
        plt.clf()
        plt.title('Training...')
    plt.xlabel('Episode')
    plt.ylabel('Duration')
    plt.plot(durations_t.numpy())
    # Take 100 episode averages and plot them too
    if len(durations_t) >= 100:
        means = durations_t.unfold(0, 100, 1).mean(1).view(-1)
        means = torch.cat((torch.zeros(99), means))
        plt.plot(means.numpy())

    plt.pause(0.001)  # pause a bit so that plots are updated

In [9]:
# traning loop 
def train_model() :
    # check if memory > batch size 
    if len(memory) < BATCH_SIZE :
        return None
    else :
        experiences = memory.sample(BATCH_SIZE)
        batch =  Experience(*zip(*experiences))

        # mask for  non terminal next_state 
        mask_terminal = torch.tensor(tuple(map(lambda x : x is not None , batch.next_state)) , device=device , dtype=torch.bool)
        non_termianl_next_state  = torch.cat([s for s in batch.next_state if s is not None])

        # convert list ( tensors)  to tensor.shape ( batch , tensor)

        state_batch = torch.cat(batch.state)
        action_batch = torch.cat(batch.action)
        reward_batch = torch.cat(batch.reward)
        
        # compute Q(s,a)  find Q(S) and choose qvalue for given action
        state_action_values   = policy_net(state_batch).gather(1 , action_batch)

        # target  q value : max A* Q(s+1 , a) * Gamma + reward 
        next_state_qvalue = torch.zeros(Batch_Size , device=device)
        # compute (Qs+1, a) for a in action  for non terminal next action
        with torch.no_grad() :
            next_state_qvalue[mask_terminal]= target_net(non_termianl_next_state).max(1).values
        excepted_state_action_values = (next_state_qvalue *GAMMA) + reward_batch

        # loss func 
        loss= torch.nn.functional.huber_loss(state_action_values , excepted_state_action_values.unsqueeze(1))

        # optimize
        optimizer.zero_grad()
        loss.backward()
        # clipping large gradient 
        torch.nn.utils.clip_grad_value_(policy_net.parameters() , 100)
        optimizer.step()

    
